# iPSC-EC day 60 — endothelial vs fibroblast marker violins

Per-gene violin plots of six endothelial and six fibroblast markers across the two
day-60 iPSC-EC media conditions (5% FBS gfp vs. VEGF+SB+3% FBS), three biological
replicates each. Violins pool the replicates per condition; individual replicates are
overlaid as dots and the inner box shows the median and quartiles.

**Pipeline**: keep genes detected in at least 3 samples -> total-count normalization to
10,000 per sample -> `log1p` -> select the marker panel -> pool the three replicates per
condition.

Unlike Figures 1E/2D/4A this notebook does not use log2/log10(FPKM+1): it applies
per-sample total-count normalization followed by `log1p`, so the y-axis is a
log-normalized expression value rather than log(FPKM+1).

**Statistics**: brackets report a two-sided Welch t-test on n = 3 vs. n = 3 biological
replicates. At this sample size only *CD34* reaches p < 0.05 and no gene survives
multiple-testing correction across the twelve markers, so the annotations are
descriptive. Set `SHOW_STATS = False` to omit them.

The replicate dots are jittered with the global NumPy RNG; `RNG` is fixed so repeat runs
are byte-identical.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
from scipy import stats
from pathlib import Path

RNG = 0
SHOW_STATS = True

OUT = Path("figures"); OUT.mkdir(exist_ok=True)
INPUT = Path("../IPSC_DAY60_batch3_expression_all_genes.csv")

In [ ]:
# the two day-60 conditions compared in this panel
conditions = {
    "sample1": "5% FBS gfp",
    "sample3": "VEGF+SB+3% FBS",
}

data = pd.read_csv(INPUT, index_col=0)
data.index = data["gene_official"]
del data["gene_official"]

# drop the conditions not shown here (sample2 = 5% FBS circ, sample4 = 3% FBS)
for s in ("sample2", "sample4"):
    for r in range(3):
        del data[f"{s}-{r}"]

data.columns = [c.replace(c.split("-")[0], conditions[c.split("-")[0]]) for c in data.columns]

adata = ad.AnnData(data.T)
adata.var_names_make_unique()
adata.obs["group"] = pd.Categorical([c.rsplit("-", 1)[0] for c in adata.obs_names])
adata

In [ ]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(adata.shape, "samples x genes")

In [ ]:
cell_markers = {
    "Endothelial Cells": ["FABP4", "GPIHBP1", "CD34", "PECAM1", "VWF", "VEGFA"],
    "Fibroblasts": ["SERPINF1", "RGS5", "MMP7", "LUM", "BGN", "PDGFRA"],
}
filtered_cell_markers = {ct: [g for g in ms if g in adata.var_names]
                         for ct, ms in cell_markers.items()}

genes = sum(filtered_cell_markers.values(), [])
order = list(adata.obs["group"].cat.categories)
PALETTE = {order[0]: "#8d8dc6", order[1]: "#7cc4a4"}   # periwinkle / mint

df = adata[:, genes].to_df()
df["group"] = adata.obs["group"].values
df.groupby("group", observed=True).size()

In [ ]:
def stars(p):
    if p < 1e-4: return "****"
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 0.05: return "*"
    return "ns"


np.random.seed(RNG)   # sns.stripplot jitters the replicate dots with the global RNG

nrow, ncol = 2, 6
fig, axes = plt.subplots(nrow, ncol, figsize=(17, 8))
rows = []

for ax, gene in zip(axes.flat, genes):
    sns.violinplot(data=df, x="group", y=gene, order=order, hue="group",
                   hue_order=order, palette=PALETTE, legend=False,
                   cut=0, inner=None, linewidth=0.8, ax=ax)
    sns.boxplot(data=df, x="group", y=gene, order=order, width=0.13,
                showcaps=False, showfliers=False, boxprops={"facecolor": "black"},
                medianprops={"color": "white", "linewidth": 1.2},
                whiskerprops={"color": "black", "linewidth": 1.2}, ax=ax)
    sns.stripplot(data=df, x="group", y=gene, order=order, color="black",
                  size=5, jitter=0.12, alpha=0.9, ax=ax)

    a = df.loc[df["group"] == order[0], gene]
    b = df.loc[df["group"] == order[1], gene]
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    rows.append({"gene": gene, f"mean_{order[0]}": a.mean(),
                 f"mean_{order[1]}": b.mean(), "n_per_group": len(a),
                 "welch_p": p, "significance": stars(p)})

    lo, hi = df[gene].min(), df[gene].max()
    span = (hi - lo) or 1.0
    if SHOW_STATS:
        y, h = hi + 0.14 * span, 0.05 * span
        ax.plot([0, 0, 1, 1], [y, y + h, y + h, y], lw=1.1, c="0.35")
        ax.text(0.5, y + h * 1.15, stars(p), ha="center", va="bottom",
                fontsize=12, fontweight="bold", color="0.15")
        ax.text(0.5, y + h * 0.05, f"p={p:.3g}", ha="center", va="top",
                fontsize=8, color="0.45")
        ax.set_ylim(lo - 0.08 * span, y + h * 5.5)
    else:
        ax.set_ylim(lo - 0.10 * span, hi + 0.10 * span)

    ax.set_title(gene, fontstyle="italic", fontsize=13, pad=8)
    ax.set_xlabel(""); ax.set_ylabel("")
    sns.despine(ax=ax)

for r in range(nrow):
    axes[r, 0].set_ylabel("log-norm expr")
    for c in range(ncol):
        if r < nrow - 1:
            axes[r, c].set_xticklabels([])
        else:
            axes[r, c].set_xticklabels(order, rotation=20, ha="right")

for r, label in enumerate(filtered_cell_markers):
    box = axes[r, 0].get_position()
    fig.text(0.005, (box.y0 + box.y1) / 2, label, rotation=90,
             va="center", ha="left", fontsize=12, fontweight="bold", color="0.25")

fig.suptitle("Endothelial vs fibroblast markers — iPSC-EC Day 60 conditions",
             fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0.025, 0, 1, 0.96])

stem = "iPSC_EC_Day60_marker_violin"
fig.savefig(OUT / f"{stem}.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT / f"{stem}.pdf", bbox_inches="tight")
pd.DataFrame(rows).to_csv(OUT / f"{stem}_stats.csv", index=False)
pd.DataFrame(rows)